In [ ]:
# Importando Bibliotecas
from openpyxl import load_workbook
import pandas as pd
import re
import unicodedata
from pathlib import Path

# =========================
# CONFIGURAÇÃO
# =========================

# Arquivo "base" que mantém layout (XLSX)
arquivo_base = r".\xlsx\Movimentacoes_01-01-2023_09-30-2025.xlsx"

# Arquivos que precisam ser incorporados (podem ser xlsx)
arquivos_adicionais = [
    r".\xlsx\Movimentacoes_10-01-2025_12-31-2025.xlsx",
]

# Saída final
saida = r"..\..\..\..\Base de Dados Original\xlsx\Movimentacoes\Movimentacoes.xlsx"


In [ ]:
# Funções para a execução
def limpar_dados_aba(ws):
    """Remove apenas os dados (mantém cabeçalho/estilos na linha 1)."""
    if ws.max_row > 1:
        ws.delete_rows(2, ws.max_row - 1)

def ler_aba_openpyxl(workbook, nome_aba):
    """Lê uma aba do openpyxl preservando texto como está (sem lower)."""
    ws = workbook[nome_aba]
    dados = []
    for row in ws.iter_rows(values_only=True):
        dados.append(list(row))

    if not dados:
        return pd.DataFrame()

    colunas = [str(c) if c is not None else f"Col_{i}" for i, c in enumerate(dados[0])]
    return pd.DataFrame(dados[1:], columns=colunas)

def _norm_txt(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def _concept(col_name: str) -> str:
    """Mapeia nomes diferentes para o mesmo 'conceito'."""
    n = _norm_txt(col_name)

    # Nº Pessoal (variações)
    if n in {"npessoal", "numpessoal", "numpess", "nopessoal", "n"} or "pessoal" in n:
        return "npessoal"

    # Data de alteração (variações)
    if "datadealteracao" in n or n in {"dataalteracao", "datamov", "data"}:
        return "dataalteracao"

    # Tipo da movimentação
    if "tipodamovimentacao" in n or "tipomovimentacao" in n:
        return "tipomovimentacao"

    # Nome
    if n in {"nome", "nomecompleto"}:
        return "nome"

    # Sistema origem
    if "sistemaorigem" in n:
        return "sistemaorigem"

    return n

def _build_concept_map(df: pd.DataFrame) -> dict:
    """Retorna dict conceito->nome_real_da_coluna (primeira ocorrência)."""
    m = {}
    for c in df.columns:
        cc = _concept(c)
        if cc not in m:
            m[cc] = c
    return m

def _clean_digits(v) -> str:
    s = "" if v is None else str(v)
    s = s.replace(".0", "")
    return re.sub(r"\D+", "", s)

def _clean_date(v) -> str:
    """Normaliza data para YYYY-MM-DD quando possível."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    s = s.replace(".", "/")
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    return "" if pd.isna(dt) else dt.strftime("%Y-%m-%d")

def _clean_text(v) -> str:
    s = "" if v is None else str(v)
    s = " ".join(s.strip().split())
    return s.lower()

def _make_key_mov(df: pd.DataFrame) -> pd.Series:
    """Chave principal: Nº Pessoal + Data de alteração + Tipo da Movimentação."""
    cmap = _build_concept_map(df)
    c_np = cmap.get("npessoal")
    c_dt = cmap.get("dataalteracao")
    c_tp = cmap.get("tipomovimentacao")

    # Se tiver o trio completo, usa ele
    if c_np and c_dt and c_tp:
        return (
            df[c_np].apply(_clean_digits) + "|"
            + df[c_dt].apply(_clean_date) + "|"
            + df[c_tp].apply(_clean_text)
        )

    # Fallbacks (caso alguma coluna falte)
    if c_np and c_dt:
        return df[c_np].apply(_clean_digits) + "|" + df[c_dt].apply(_clean_date)

    if c_np and c_tp:
        return df[c_np].apply(_clean_digits) + "|" + df[c_tp].apply(_clean_text)

    # Último fallback (evita quebrar)
    return df.astype(str).agg("|".join, axis=1)

def _assinatura_arquivo(caminho: str, n: int = 64) -> bytes:
    with open(caminho, "rb") as f:
        return f.read(n)

def ler_arquivo_excel_para_dfs(caminho: str) -> list[tuple[str, pd.DataFrame]]:
    """
    Lê xlsx/xls (mesmo se estiver com extensão errada) e também HTML disfarçado de .xls.
    Retorna lista de (nome_aba, df).
    """
    sig = _assinatura_arquivo(caminho, 64)
    sig_strip = sig.lstrip()

    # XLSX (arquivo zip): começa com "PK"
    if sig.startswith(b"PK\x03\x04"):
        xls = pd.ExcelFile(caminho, engine="openpyxl")
        return [(aba, pd.read_excel(xls, sheet_name=aba)) for aba in xls.sheet_names]

    # XLS antigo (OLE): começa com D0 CF 11 E0...
    if sig.startswith(b"\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1"):
        # precisa do xlrd instalado: pip install xlrd==2.0.1
        xls = pd.ExcelFile(caminho, engine="xlrd")
        return [(aba, pd.read_excel(xls, sheet_name=aba)) for aba in xls.sheet_names]

    # HTML disfarçado (muito comum em export do SAP)
    if sig_strip.startswith(b"<") or b"<html" in sig.lower() or b"<!doctype" in sig.lower():
        # pega a primeira tabela do HTML
        tabelas = pd.read_html(caminho)
        df = tabelas[0] if tabelas else pd.DataFrame()
        return [("HTML_TABLE", df)]

    # Se cair aqui, não é Excel nem HTML reconhecível
    raise RuntimeError(
        f"Formato não reconhecido em '{caminho}'. "
        f"Abra no Excel e 'Salvar como' .xlsx para padronizar."
    )

def alinhar_colunas_por_conceito(df_extra: pd.DataFrame, df_dest: pd.DataFrame) -> pd.DataFrame:
    """Renomeia colunas do extra para bater com os conceitos do destino."""
    cmap_dest = _build_concept_map(df_dest)
    renames = {}
    for col in df_extra.columns:
        c = _concept(col)
        if c in cmap_dest and col != cmap_dest[c]:
            renames[col] = cmap_dest[c]
    if renames:
        df_extra = df_extra.rename(columns=renames)
    return df_extra

def escolher_aba_destino_por_intersecao(df_extra: pd.DataFrame, wb_principal):
    """
    Escolhe a aba do WB que tem maior interseção de colunas (conceitos) com o df_extra.
    Se seu arquivo final tiver só 1 aba, ele naturalmente vai escolher ela.
    """
    extra_concepts = set(_concept(c) for c in df_extra.columns)

    melhor_idx = 0
    melhor_score = -1

    for idx, nome_aba in enumerate(wb_principal.sheetnames):
        df_dest = ler_aba_openpyxl(wb_principal, nome_aba)
        if df_dest.empty:
            continue

        dest_concepts = set(_concept(c) for c in df_dest.columns)
        inter = len(extra_concepts & dest_concepts)
        score = inter / max(1, len(extra_concepts))

        if score > melhor_score:
            melhor_score = score
            melhor_idx = idx

    return melhor_idx

def inserir_novos_na_aba(wb_principal, idx_destino: int, df_extra: pd.DataFrame):
    nome_aba = wb_principal.sheetnames[idx_destino]
    ws = wb_principal[nome_aba]

    df_dest = ler_aba_openpyxl(wb_principal, nome_aba)
    if df_dest.empty:
        # se estiver vazio, cria destino com colunas do extra
        df_dest = pd.DataFrame(columns=list(df_extra.columns))

    # 1) alinhar por conceito (ex.: "Nº Pessoal" vs "Nº Pessoal ")
    df_extra = alinhar_colunas_por_conceito(df_extra, df_dest)

    # 2) garantir união de colunas (ordem do destino primeiro)
    colunas = list(df_dest.columns)
    for c in df_extra.columns:
        if c not in colunas:
            colunas.append(c)

    df_dest = df_dest.reindex(columns=colunas)
    df_extra = df_extra.reindex(columns=colunas)

    # 3) criar chave e filtrar só novos
    df_dest["_key"] = _make_key_mov(df_dest)
    df_extra["_key"] = _make_key_mov(df_extra)

    keys_dest = set(df_dest["_key"].dropna())
    novos = df_extra[~df_extra["_key"].isin(keys_dest)].copy()

    print(f"➕ Aba '{nome_aba}': {len(novos)} linha(s) nova(s) inserida(s) (de {len(df_extra)} no arquivo adicional).")

    # 4) concat e remover auxiliares
    df_final = pd.concat([df_dest, novos], ignore_index=True)
    df_final = df_final.drop(columns=["_key"])

    # 5) reescrever mantendo cabeçalho/estilos
    limpar_dados_aba(ws)
    for r_idx, row in enumerate(df_final.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)


In [ ]:
# Execução
print("📌 Abrindo arquivo base:", arquivo_base)
wb = load_workbook(arquivo_base)
print("✅ Abas no arquivo base:", wb.sheetnames)

for caminho in arquivos_adicionais:
    print(f"\n📥 Lendo adicional: {caminho}")
    abas_dfs = ler_arquivo_excel_para_dfs(caminho)

    for nome_aba_extra, df_extra in abas_dfs:
        if df_extra.empty:
            continue

        # escolhe melhor aba destino no arquivo final (normalmente só 1)
        idx_dest = escolher_aba_destino_por_intersecao(df_extra, wb)
        inserir_novos_na_aba(wb, idx_dest, df_extra)

# salvar
Path(saida).parent.mkdir(parents=True, exist_ok=True)
wb.save(saida)
print(f"\n✅ Movimentações consolidadas salvas em: {saida}")
